# 🎯 ATS Resume Scoring System

## Project Overview

This notebook implements a fully functional **Applicant Tracking System (ATS) Resume Scorer** using pretrained NLP models. It evaluates a resume PDF against a job description and produces a structured score out of 100.

### Score Breakdown
| Component | Max Score |
|---|---|
| Keyword Match | 40 |
| Semantic Similarity | 30 |
| Skills Coverage | 10 |
| Experience Relevance | 15 |
| Formatting Quality | 5 |
| **Total** | **100** |

### Tech Stack
- `pdfplumber` — PDF text extraction
- `spaCy` — NLP tokenization, entity recognition, noun chunks
- `sentence-transformers` — Semantic similarity via `all-MiniLM-L6-v2`
- `scikit-learn` — Cosine similarity computation
- `matplotlib` — Score visualization
- No paid APIs. Fully offline-capable after model download.

## 1. Installing Dependencies

## 2. Importing Libraries

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pdfplumber
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# Load sentence transformer model (downloads once, cached thereafter)
print("Loading sentence transformer model...")
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ All libraries loaded.")

## 3. Resume Parsing

Extract and clean text from a resume PDF using `pdfplumber`. Special characters are removed and text is normalized.

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract raw text from a PDF file using pdfplumber.

    Args:
        pdf_path: Path to the PDF file.

    Returns:
        Raw text extracted from the PDF.
    """
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"❌ Error reading PDF: {e}")
        raise
    return text


def clean_text(text: str) -> str:
    """
    Clean and normalize raw text extracted from a resume.

    Steps:
      - Remove non-ASCII and control characters
      - Normalize whitespace
      - Lowercase the text

    Args:
        text: Raw text string.

    Returns:
        Cleaned, lowercased text.
    """
    # Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # Remove special bullets and unicode-like chars
    text = re.sub(r'[\x80-\xFF]', ' ', text)
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Lowercase
    text = text.lower()
    return text


def has_tables(pdf_path: str) -> bool:
    """
    Detect whether the PDF contains table structures.

    Args:
        pdf_path: Path to the PDF file.

    Returns:
        True if tables are detected, False otherwise.
    """
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                if page.extract_tables():
                    return True
    except Exception:
        pass
    return False


# ─── Demo: Parse the uploaded resume ─────────────────────────────────────────
RESUME_PDF_PATH = "/mnt/user-data/uploads/10265057.pdf"  # ← change to your PDF path

raw_resume_text = extract_text_from_pdf(RESUME_PDF_PATH)
clean_resume_text = clean_text(raw_resume_text)
resume_has_tables = has_tables(RESUME_PDF_PATH)

print(f"✅ Resume parsed successfully.")
print(f"   Total characters : {len(clean_resume_text)}")
print(f"   Word count       : {len(clean_resume_text.split())}")
print(f"   Tables detected  : {resume_has_tables}")
print("\n--- Resume Preview (first 500 chars) ---")
print(clean_resume_text[:500])

## 4. Job Description Processing

Define the target job description. In production this can be loaded from a text file or input widget.

In [ ]:
# ─── Paste or load the Job Description here ───────────────────────────────────
JOB_DESCRIPTION = """
We are looking for a Senior RF Systems Engineer to join our satellite communications team.

Responsibilities:
- Design and develop RF hardware units including antennas, receivers, filters, and amplifiers.
- Generate cascade RF performance analyses including NF, Gain, IP3, 1dB compression, and spurious.
- Lead EVT, DVT, and PVT verification and validation processes.
- Define DFM procedures and requirements for subcontractors and suppliers.
- Perform Failure Analysis (FA) to drive root cause from system to unit level.
- Collaborate with cross-functional teams on satellite system design.
- Work with SQL databases and data analysis tools for system performance monitoring.
- Utilize spectrum analyzers, vector network analyzers, and signal analyzers.

Requirements:
- M.S. in Electrical Engineering or related field.
- 5+ years of experience in RF systems engineering.
- Proficiency in Python, MATLAB, C/C++.
- Experience with AWR Microwave Office or Ansoft Designer.
- Strong background in satellite communication systems.
- Familiarity with PCB design and SPICE simulation.
- Knowledge of LabVIEW, SQL, and data analysis.
- Excellent written and verbal communication skills.
"""

clean_jd_text = clean_text(JOB_DESCRIPTION)
print(f"✅ Job description loaded.")
print(f"   Word count: {len(clean_jd_text.split())}")

## 5. Keyword Extraction

We use spaCy to extract meaningful keywords from the job description by:
- Removing stopwords and punctuation
- Extracting noun chunks and named entities
- Filtering tokens by part-of-speech (NOUN, PROPN, ADJ)

In [ ]:
def extract_keywords_spacy(text: str, top_n: int = 60) -> list:
    """
    Extract important keywords from text using spaCy.

    Strategy:
      1. Token-level: nouns, proper nouns, adjectives (non-stopword, alphabetic)
      2. Noun chunks (multi-word phrases)
      3. Named entities

    Args:
        text: Input text (already cleaned/lowercased).
        top_n: Maximum keywords to return.

    Returns:
        Deduplicated list of keyword strings.
    """
    doc = nlp(text)
    keywords = set()

    # Single-token keywords
    for token in doc:
        if (
            not token.is_stop
            and not token.is_punct
            and token.is_alpha
            and len(token.text) > 2
            and token.pos_ in ('NOUN', 'PROPN', 'ADJ')
        ):
            keywords.add(token.lemma_.lower())

    # Noun chunks (multi-word)
    for chunk in doc.noun_chunks:
        chunk_text = chunk.text.strip().lower()
        if len(chunk_text.split()) <= 4 and len(chunk_text) > 3:
            keywords.add(chunk_text)

    # Named entities
    for ent in doc.ents:
        if ent.label_ not in ('DATE', 'TIME', 'CARDINAL', 'ORDINAL', 'PERCENT'):
            keywords.add(ent.text.strip().lower())

    return list(keywords)[:top_n]


jd_keywords = extract_keywords_spacy(clean_jd_text)
print(f"✅ Extracted {len(jd_keywords)} keywords from job description.")
print("\nSample keywords:")
print(jd_keywords[:20])

## 6. Keyword Match Score (0–40)

**Formula:** `keyword_score = (matched_keywords / total_keywords) * 40`

Each JD keyword is checked for presence in the cleaned resume text.

In [ ]:
def compute_keyword_score(resume_text: str, jd_keywords: list, max_score: float = 40.0) -> dict:
    """
    Compute keyword match score.

    Args:
        resume_text: Cleaned resume text.
        jd_keywords: List of keywords from job description.
        max_score: Maximum possible score for this component.

    Returns:
        Dict with score, matched, missing, and total keyword counts.
    """
    matched = [kw for kw in jd_keywords if kw in resume_text]
    missing = [kw for kw in jd_keywords if kw not in resume_text]
    total = len(jd_keywords)

    if total == 0:
        return {'score': 0.0, 'matched': [], 'missing': [], 'total': 0}

    raw_score = (len(matched) / total) * max_score
    return {
        'score': round(raw_score, 2),
        'matched': matched,
        'missing': missing,
        'total': total
    }


keyword_result = compute_keyword_score(clean_resume_text, jd_keywords)
print(f"✅ Keyword Match Score : {keyword_result['score']} / 40")
print(f"   Matched Keywords   : {len(keyword_result['matched'])} / {keyword_result['total']}")
print(f"\n   Missing keywords (top 15):")
print(keyword_result['missing'][:15])

## 7. Semantic Similarity Score (0–30)

Use `sentence-transformers` with model `all-MiniLM-L6-v2` to compute dense vector embeddings of the resume and JD, then compute cosine similarity.

**Formula:** `semantic_score = cosine_similarity(resume_emb, jd_emb) * 30`

In [ ]:
def compute_semantic_score(
    resume_text: str,
    jd_text: str,
    model: SentenceTransformer,
    max_score: float = 30.0
) -> dict:
    """
    Compute semantic similarity score using sentence-transformers.

    Args:
        resume_text: Cleaned resume text.
        jd_text: Cleaned job description text.
        model: Loaded SentenceTransformer model.
        max_score: Maximum possible score for this component.

    Returns:
        Dict with score and raw cosine similarity.
    """
    # Truncate to ~512 tokens worth of characters for speed
    resume_snippet = resume_text[:3000]
    jd_snippet = jd_text[:3000]

    resume_emb = model.encode([resume_snippet])
    jd_emb = model.encode([jd_snippet])

    similarity = cosine_similarity(resume_emb, jd_emb)[0][0]
    score = float(similarity) * max_score

    return {
        'score': round(score, 2),
        'cosine_similarity': round(float(similarity), 4)
    }


semantic_result = compute_semantic_score(clean_resume_text, clean_jd_text, sentence_model)
print(f"✅ Semantic Similarity Score : {semantic_result['score']} / 30")
print(f"   Cosine Similarity        : {semantic_result['cosine_similarity']}")

## 8. Skills Coverage Score (0–10)

Extract a curated skill list from the JD using regex patterns for known technical terms, programming languages, tools, and frameworks. Then check how many appear in the resume.

In [ ]:
# Common tech/engineering skill patterns (expand as needed)
SKILL_PATTERNS = [
    r'\bpython\b', r'\bmatlab\b', r'\bc\+\+\b', r'\bc/c\+\+\b',
    r'\bjava\b', r'\bsql\b', r'\blabview\b', r'\bspice\b',
    r'\bpcb\b', r'\brf\b', r'\bfpga\b', r'\bvhdl\b', r'\bverilog\b',
    r'\bawr\b', r'\bansoft\b', r'\bsas\b', r'\bspss\b',
    r'\bexcel\b', r'\bnumpy\b', r'\bpandas\b', r'\btensorflow\b',
    r'\bkeras\b', r'\btorch\b', r'\bscikit\b', r'\bgit\b',
    r'\bdocker\b', r'\blinux\b', r'\bwindows\b', r'\baws\b',
    r'\bgcp\b', r'\bazure\b',
]


def extract_skills_from_text(text: str, skill_patterns: list) -> list:
    """
    Extract skills present in the given text using regex patterns.

    Args:
        text: Text to search in.
        skill_patterns: List of regex patterns for skills.

    Returns:
        List of matched skill strings.
    """
    found = []
    for pattern in skill_patterns:
        if re.search(pattern, text, re.IGNORECASE):
            clean_skill = pattern.replace(r'\b', '').replace('\\', '')
            found.append(clean_skill)
    return list(set(found))


def compute_skills_score(
    resume_text: str,
    jd_text: str,
    skill_patterns: list,
    max_score: float = 10.0
) -> dict:
    """
    Compute skill coverage score.

    Args:
        resume_text: Cleaned resume text.
        jd_text: Cleaned JD text.
        skill_patterns: Regex skill patterns to look for.
        max_score: Maximum possible score.

    Returns:
        Dict with score, jd_skills, resume_skills, and missing_skills.
    """
    jd_skills = extract_skills_from_text(jd_text, skill_patterns)
    resume_skills = extract_skills_from_text(resume_text, skill_patterns)

    if not jd_skills:
        return {'score': max_score, 'jd_skills': [], 'resume_skills': [], 'missing_skills': []}

    matched = [s for s in jd_skills if s in resume_skills]
    missing = [s for s in jd_skills if s not in resume_skills]
    coverage = len(matched) / len(jd_skills)
    score = coverage * max_score

    return {
        'score': round(score, 2),
        'jd_skills': jd_skills,
        'resume_skills': resume_skills,
        'matched_skills': matched,
        'missing_skills': missing
    }


skills_result = compute_skills_score(clean_resume_text, clean_jd_text, SKILL_PATTERNS)
print(f"✅ Skills Coverage Score : {skills_result['score']} / 10")
print(f"   JD Skills Detected   : {skills_result['jd_skills']}")
print(f"   Matched in Resume    : {skills_result['matched_skills']}")
print(f"   Missing from Resume  : {skills_result['missing_skills']}")

## 9. Experience Relevance Score (0–15)

Use regex to:
1. Extract years of experience claimed in the resume
2. Extract required years from the JD
3. Score proportionally (capped at max)

In [ ]:
def extract_years_of_experience(text: str) -> float:
    """
    Extract years of experience from text using regex heuristics.

    Searches for patterns like:
      - "5 years of experience"
      - "5+ years"
      - "five years"

    Falls back to estimating from employment date ranges.

    Args:
        text: Cleaned text (resume or JD).

    Returns:
        Estimated years as a float. Returns 0.0 if not found.
    """
    patterns = [
        r'(\d+)\+?\s*years?\s+of\s+experience',
        r'(\d+)\+?\s*years?\s+experience',
        r'experience\s+of\s+(\d+)\+?\s*years?',
        r'(\d+)\+?\s*years?\s+in\s+',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return float(match.group(1))
    return 0.0


def estimate_experience_from_dates(text: str) -> float:
    """
    Estimate years of experience by finding year ranges in resume text.

    Args:
        text: Resume text.

    Returns:
        Total estimated years as float.
    """
    # Match patterns like "2011 to 2014", "2014 - current", "may 2014 to current"
    year_pattern = r'(\d{4})\s*(?:to|-)\s*(\d{4}|current|present)'
    matches = re.findall(year_pattern, text, re.IGNORECASE)
    total_years = 0.0
    import datetime
    current_year = datetime.datetime.now().year
    for start, end in matches:
        start_yr = int(start)
        end_yr = current_year if end.lower() in ('current', 'present') else int(end)
        total_years += max(0, end_yr - start_yr)
    return total_years


def compute_experience_score(
    resume_text: str,
    jd_text: str,
    max_score: float = 15.0
) -> dict:
    """
    Compute experience relevance score.

    Args:
        resume_text: Cleaned resume text.
        jd_text: Cleaned JD text.
        max_score: Maximum score.

    Returns:
        Dict with score, required_years, resume_years.
    """
    required_years = extract_years_of_experience(jd_text)

    # Try explicit mention first, then estimate from dates
    resume_years = extract_years_of_experience(resume_text)
    if resume_years == 0.0:
        resume_years = estimate_experience_from_dates(resume_text)

    if required_years == 0:
        # No requirement stated; give full marks if any experience detected
        score = max_score if resume_years > 0 else max_score * 0.5
    else:
        ratio = min(resume_years / required_years, 1.0)
        score = ratio * max_score

    return {
        'score': round(score, 2),
        'required_years': required_years,
        'resume_years': resume_years
    }


experience_result = compute_experience_score(clean_resume_text, clean_jd_text)
print(f"✅ Experience Relevance Score : {experience_result['score']} / 15")
print(f"   Required Years (JD)       : {experience_result['required_years']}")
print(f"   Detected Years (Resume)   : {experience_result['resume_years']}")

## 10. Formatting Score (0–5)

Penalize resumes that:
- Contain tables (ATS parsers often struggle)
- Have too many special characters (> 5% of total characters)
- Are too short (< 300 words)

In [ ]:
def compute_formatting_score(
    raw_text: str,
    has_tables_flag: bool,
    max_score: float = 5.0
) -> dict:
    """
    Compute formatting quality score.

    Penalties applied for:
      - Tables detected      → -2 points
      - Too many special chars → -1 point
      - Resume too short     → -2 points

    Args:
        raw_text: Raw (uncleaned) resume text.
        has_tables_flag: True if PDF contains tables.
        max_score: Maximum score.

    Returns:
        Dict with score and penalty details.
    """
    score = max_score
    penalties = []

    # Penalty 1: Tables detected
    if has_tables_flag:
        score -= 2
        penalties.append("Tables detected (-2): ATS may misparse tabular content.")

    # Penalty 2: Excessive special characters
    total_chars = len(raw_text)
    special_chars = len(re.findall(r'[^a-zA-Z0-9\s.,;:\-/()]', raw_text))
    if total_chars > 0 and (special_chars / total_chars) > 0.05:
        score -= 1
        penalties.append(f"Excess special chars (-1): {special_chars}/{total_chars} ({100*special_chars/total_chars:.1f}%)")

    # Penalty 3: Too short
    word_count = len(raw_text.split())
    if word_count < 300:
        score -= 2
        penalties.append(f"Resume too short (-2): {word_count} words (min 300).")

    score = max(0.0, score)  # Floor at 0

    return {
        'score': round(score, 2),
        'word_count': word_count,
        'has_tables': has_tables_flag,
        'penalties': penalties if penalties else ['No penalties — formatting looks good!']
    }


formatting_result = compute_formatting_score(raw_resume_text, resume_has_tables)
print(f"✅ Formatting Score : {formatting_result['score']} / 5")
print(f"   Word Count     : {formatting_result['word_count']}")
for p in formatting_result['penalties']:
    print(f"   → {p}")

## 11. ATS Scoring Engine — Final Score

Aggregate all component scores into the final ATS score, capped at 100.

In [ ]:
def compute_ats_score(
    keyword_result: dict,
    semantic_result: dict,
    skills_result: dict,
    experience_result: dict,
    formatting_result: dict
) -> dict:
    """
    Compute the final ATS score from all component scores.

    Args:
        keyword_result: Output of compute_keyword_score().
        semantic_result: Output of compute_semantic_score().
        skills_result: Output of compute_skills_score().
        experience_result: Output of compute_experience_score().
        formatting_result: Output of compute_formatting_score().

    Returns:
        Dict with final_score, breakdown, and grade.
    """
    breakdown = {
        'Keyword Match (40)': keyword_result['score'],
        'Semantic Similarity (30)': semantic_result['score'],
        'Skills Coverage (10)': skills_result['score'],
        'Experience Match (15)': experience_result['score'],
        'Formatting (5)': formatting_result['score'],
    }
    total = sum(breakdown.values())
    final_score = min(round(total, 2), 100.0)  # Cap at 100

    # Grade assignment
    if final_score >= 85:
        grade = 'Excellent ✅'
    elif final_score >= 70:
        grade = 'Good 🟡'
    elif final_score >= 50:
        grade = 'Average 🟠'
    else:
        grade = 'Needs Improvement 🔴'

    return {
        'final_score': final_score,
        'breakdown': breakdown,
        'grade': grade
    }


ats_result = compute_ats_score(
    keyword_result,
    semantic_result,
    skills_result,
    experience_result,
    formatting_result
)

# ─── Print Final Report ────────────────────────────────────────────────────────
print("=" * 50)
print(f"  ATS SCORE: {ats_result['final_score']} / 100  |  {ats_result['grade']}")
print("=" * 50)
print("\nBreakdown:")
for component, score in ats_result['breakdown'].items():
    print(f"  {component:<30} : {score}")
print("=" * 50)

## 12. Resume Improvement Suggestions

Based on missing keywords and skills, provide actionable recommendations.

In [ ]:
def generate_suggestions(
    keyword_result: dict,
    skills_result: dict,
    experience_result: dict,
    formatting_result: dict,
    top_n_keywords: int = 10
) -> None:
    """
    Print actionable suggestions for improving the resume's ATS score.

    Args:
        keyword_result: Keyword scoring result dict.
        skills_result: Skills scoring result dict.
        experience_result: Experience scoring result dict.
        formatting_result: Formatting scoring result dict.
        top_n_keywords: How many missing keywords to show.
    """
    print("\n📋 RESUME IMPROVEMENT SUGGESTIONS")
    print("-" * 50)

    # Keyword suggestions
    missing_kw = keyword_result.get('missing', [])[:top_n_keywords]
    if missing_kw:
        print(f"\n🔑 Add these missing JD keywords to your resume:")
        for kw in missing_kw:
            print(f"   • {kw}")

    # Skills suggestions
    missing_skills = skills_result.get('missing_skills', [])
    if missing_skills:
        print(f"\n🛠️  Add these missing skills (found in JD, not in resume):")
        for sk in missing_skills:
            print(f"   • {sk}")

    # Experience suggestions
    req_yrs = experience_result.get('required_years', 0)
    res_yrs = experience_result.get('resume_years', 0)
    if req_yrs > 0 and res_yrs < req_yrs:
        print(f"\n📅 Experience Gap: JD requires {req_yrs} years, resume shows ~{res_yrs:.0f} years.")
        print("    → Highlight all relevant work, internships, and project experience.")

    # Formatting suggestions
    for penalty in formatting_result.get('penalties', []):
        if '-' in penalty:
            print(f"\n📄 Formatting: {penalty}")
            if 'table' in penalty.lower():
                print("    → Replace tables with plain bullet lists for ATS compatibility.")
            elif 'short' in penalty.lower():
                print("    → Expand experience descriptions, add more detail to projects.")
            elif 'special' in penalty.lower():
                print("    → Remove decorative symbols, use standard ASCII characters.")

    print("\n✅ General Tips:")
    print("   • Mirror the exact language used in the job description.")
    print("   • Quantify achievements where possible (e.g., 'reduced latency by 20%').")
    print("   • Keep formatting clean and ATS-parseable (no graphics, no columns).")
    print("   • Use standard section headings: Experience, Education, Skills.")


generate_suggestions(keyword_result, skills_result, experience_result, formatting_result)

## 13. Visualization of Score Breakdown

A horizontal bar chart showing each component's achieved score vs its maximum.

In [ ]:
def plot_ats_breakdown(ats_result: dict) -> None:
    """
    Plot a horizontal bar chart of ATS score components.

    Args:
        ats_result: Output of compute_ats_score().
    """
    breakdown = ats_result['breakdown']
    max_scores = [40, 30, 10, 15, 5]  # Corresponds to component order

    labels = list(breakdown.keys())
    achieved = list(breakdown.values())
    remaining = [m - a for m, a in zip(max_scores, achieved)]

    fig, ax = plt.subplots(figsize=(10, 5))

    bars_achieved = ax.barh(labels, achieved, label='Score Achieved')
    bars_remaining = ax.barh(labels, remaining, left=achieved, label='Score Remaining', alpha=0.35)

    # Annotate achieved scores
    for bar, val, max_val in zip(bars_achieved, achieved, max_scores):
        ax.text(
            bar.get_width() / 2,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}/{max_val}",
            va='center', ha='center', fontsize=9, fontweight='bold', color='white'
        )

    ax.set_xlabel('Score', fontsize=11)
    ax.set_title(
        f'ATS Resume Score Breakdown\nFinal Score: {ats_result["final_score"]}/100  |  {ats_result["grade"]}',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlim(0, 45)
    ax.legend(loc='lower right', fontsize=9)
    ax.invert_yaxis()  # Highest weight component at top
    plt.tight_layout()
    plt.show()


plot_ats_breakdown(ats_result)

## 14. Full Pipeline — Run on Any Resume

This cell wraps the entire pipeline into a single callable function for convenience.

In [ ]:
def score_resume(pdf_path: str, job_description: str, verbose: bool = True) -> dict:
    """
    End-to-end ATS resume scoring pipeline.

    Args:
        pdf_path: Path to the resume PDF.
        job_description: Raw job description text.
        verbose: If True, print full report and plot.

    Returns:
        Complete result dict with all component scores and final ATS score.
    """
    # Step 1: Parse
    raw_resume = extract_text_from_pdf(pdf_path)
    clean_resume = clean_text(raw_resume)
    clean_jd = clean_text(job_description)
    tables_flag = has_tables(pdf_path)

    # Step 2–3: Keywords
    keywords = extract_keywords_spacy(clean_jd)
    kw_res = compute_keyword_score(clean_resume, keywords)

    # Step 4: Semantic similarity
    sem_res = compute_semantic_score(clean_resume, clean_jd, sentence_model)

    # Step 5: Skills
    sk_res = compute_skills_score(clean_resume, clean_jd, SKILL_PATTERNS)

    # Step 6: Experience
    exp_res = compute_experience_score(clean_resume, clean_jd)

    # Step 7: Formatting
    fmt_res = compute_formatting_score(raw_resume, tables_flag)

    # Step 8: Final ATS score
    ats_res = compute_ats_score(kw_res, sem_res, sk_res, exp_res, fmt_res)

    if verbose:
        print("=" * 50)
        print(f"  ATS SCORE: {ats_res['final_score']} / 100  |  {ats_res['grade']}")
        print("=" * 50)
        for component, score in ats_res['breakdown'].items():
            print(f"  {component:<30} : {score}")
        print("=" * 50)
        generate_suggestions(kw_res, sk_res, exp_res, fmt_res)
        plot_ats_breakdown(ats_res)

    return {
        'ats': ats_res,
        'keyword': kw_res,
        'semantic': sem_res,
        'skills': sk_res,
        'experience': exp_res,
        'formatting': fmt_res
    }


# ─── Run the full pipeline on the uploaded resume ────────────────────────────
results = score_resume(RESUME_PDF_PATH, JOB_DESCRIPTION)

## 15. Limitations of ATS Scoring

This system provides a **reasonable approximation** of ATS behavior, but it is important to understand its limitations:

### ⚠️ Known Limitations

1. **Keyword matching is lexical, not contextual.** A keyword like "python" scores the same whether you're a Python expert or you mention it once in a passing sentence.

2. **ATS systems vary widely.** Real ATS software (Taleo, Workday, Greenhouse, Lever) each have proprietary scoring algorithms. There is no universal ATS standard.

3. **Experience estimation is fragile.** The regex-based year extraction can fail for non-standard date formats, career gaps, or part-time roles.

4. **Semantic similarity is not a perfect proxy for fit.** Two highly similar texts in embedding space may not imply a good candidate-role match in practice.

5. **Formatting evaluation is limited.** Complex formatting issues (columns, text boxes, headers/footers with key info) cannot be fully captured by pdfplumber alone.

6. **No handling of synonyms by default.** "ML" and "machine learning" are not matched by keyword overlap unless both appear in the text.

7. **Domain-specific skill lists need tuning.** The hardcoded `SKILL_PATTERNS` list must be tailored to specific industries for best results.

### 💡 Potential Improvements
- Use a fine-tuned model on resume–JD pairs for semantic scoring
- Integrate synonym expansion via WordNet or domain ontologies
- Use section-level analysis (experience section vs. skills section vs. education)
- Add support for multi-page resume analysis and page-order weighting
- Train a lightweight classifier on labeled ATS pass/fail data